NicheNet’s ligand activity analysis on a gene set of interest: predict
active ligands and their target genes
================

This vignette follows the steps described in [Perform NicheNet analysis: step-by-step analysis](steps.ipynb)
with two major differences: a predefined gene set of interest is given,
and a different definition of expressed genes.

Here, we use explore intercellular communication in the tumor
microenvironment of head and neck squamous cell carcinoma (HNSCC) (Puram
et al. 2017). More specifically, we will look at which ligands expressed
by cancer-associated fibroblasts (CAFs) can induce a specific gene
program in neighboring malignant cells. The original authors of the
study have linked this partial epithelial-mesenschymal transition
(p-EMT) program to metastasis.

The used [ligand-target matrix](https://doi.org/10.5281/zenodo.7074290)
and example [expression data](https://doi.org/10.5281/zenodo.3260758) of
interacting cells can be downloaded from Zenodo.

# Prepare NicheNet analysis

### Load packages

In [1]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import LigandReceptorNetwork, WeightedNetwork
from nichenetpy.utils import (
    read_matrix_from_csv,
    read_csv_rows,
    read_list_from_csv,
    combine_by_key,
    combine_dicts,
    subset_matrix
)
from nichenetpy.extraction import (
    subset_ann_celltype,
    get_weighted_ligand_receptor_links,
    get_lfc_celltype
)
from nichenetpy.gene_symbol import human_alias_info
from nichenetpy.visualization import (
    prepare_ligand_target_visualization,
    prepare_ligand_receptor_visualization,
    heatmap_2d,
    heatmap_1d
)

from itertools import cycle, chain
from math import log

import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

## Read in NicheNet's networks

The ligand-target prior model is a matrix describing the potential that a ligand may regulate a target gene, and it is used to run the ligand activity analysis. The ligand-receptor network contains information on potential ligand-receptor bindings, and it is used to identify potential ligands. 

In [2]:
# set the paths to that of your local copies (these files are too large to upload to the remote git repo)
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/model/human/csv/ligand_target_matrix.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/model/human/csv/lr_network.csv")

### Read in the expression data of interacting cells

This is publicly available single-cell data from CAF and malignant cells
from HNSCC tumors.

In [3]:
# set the paths to that of your local copies (these files are too large to upload to the remote git repo)
exp_mat, rows, cols = read_matrix_from_csv("D:/Data/hnscc/expression.csv")
cols = human_alias_info.alias_to_symbol(cols)
geneset = read_list_from_csv("D:/Data/hnscc/expressed_genes.csv")
sample_info_col_names, sample_info = read_csv_rows("D:/Data/hnscc/sample_info.csv")

## 1. Define a set of potential ligands

Our research question is to prioritize which ligands expressed by CAFs
can induce p-EMT in neighboring malignant cells. Hence, we will only use
on the **sender-focused** approach, with CAFs as senders and malignant
cells as receivers.

The set of potential ligands is defined as ligands that are expressed in
sender cells whose cognate receptors are also expressed in receiver
cells.

So first, we will determine which genes are expressed in the sender
cells (CAFs) and receiver cells (malignant cells). We will only consider
samples from high quality primary tumors and also remove samples from
lymph node metastases. We will use the definition of expressed genes by
the original authors, that is, the aggregate expression of each gene $i$
across the $k$ cells, calculated as
$E_a(i) = log_{2}(average(TPM(i)1…k)+1)$, should be \>= 4.

We recommend users to define expressed genes in the way that they
consider to be most appropriate for their dataset. For single-cell data
generated by the 10x platform in our lab, we consider genes to be
expressed in a cell type when they have non-zero values in a certain
fraction of the cells from that cell type (usually 10%). This is used in
the vignette [Perform NicheNet analysis:
step-by-step analysis](steps.ipynb).

In [4]:
print(sample_info_col_names)
sample_info = [
    [int(processed_by_Maxima_enzyme), int(Lymph_node), int(classified_as_cancer_cell), int(classified_as_non_cancer_cells), non_cancer_cell_type, cell, tumor]
    for processed_by_Maxima_enzyme, Lymph_node, classified_as_cancer_cell, classified_as_non_cancer_cells, non_cancer_cell_type, cell, tumor
    in sample_info
]

['processed by Maxima enzyme', 'Lymph node', 'classified  as cancer cell', 'classified as non-cancer cells', 'non-cancer cell type', 'cell', 'tumor']


In [5]:
tumors_remove = ["HN10","HN","HN12", "HN13", "HN24", "HN7", "HN8","HN23"]
CAF_cells = [e[5] for e in sample_info if e[1] == 0 and e[4] == "CAF" and e[6] not in tumors_remove]
malignant_cells = [e[5] for e in sample_info if e[1] == 0 and e[2] == 1 and e[6] not in tumors_remove]
row2id = dict(zip(rows, range(len(rows))))

def get_exp(mat, cols):
    agg_exp = [
        log(sum((10*(2**x - 1) for x in mat[:, i]))/mat.shape[0] + 1, 2)
        for i in range(mat.shape[1])
    ]
    return {gene for gene, x in zip(cols, agg_exp) if x > 4}

expressed_genes_sender = get_exp(subset_matrix(exp_mat, [row2id[cell] for cell in CAF_cells]), cols)
expressed_genes_receiver = get_exp(subset_matrix(exp_mat, [row2id[cell] for cell in malignant_cells]), cols)

In [6]:
len(expressed_genes_sender)

6706

In [7]:
len(expressed_genes_receiver)

6351

Now, we can filter the expressed ligands and receptors to only those that putatively bind together. This information is stored in NicheNet’s ligand-receptor network by gathering various data sources.

In [10]:
ligands = lr_network.get_ligands()
expressed_ligands = ligands.intersection(expressed_genes_sender)
receptors = lr_network.get_receptors()
expressed_receptors = ligands.intersection(expressed_genes_receiver)
potential_ligands = {ligand for ligand, receptor in lr_network if ligand in expressed_ligands and receptor in expressed_receptors}